# 52b. Part-one fixes from the cached files: the computations the hard review asked for, no GPU

The hard review of part one (10 October) listed computations that need the cached files but no model. This notebook does them, in the order the paper reports the results, and writes one CSV per item under `reports/part1_fixes/`. Nothing here reruns a model on new text; the one scoring step (Stage B, PIGuard and the conventional fine-tune on the 455 cached adversary variants) is two encoder classifiers on 455 documents, which a CPU does in a few minutes.

**Stages.**

- **A. Section 8 under the paper's final rule.** Tables 20 to 22 were scored by the first rule (any emoji, any decodable base64 run, scam markers, the judge). The paper's final rule (notebook 46, `final_scoring.paper_verdict`) does not count emoji or base64 strings already present in the executed document, and decides translation by the judge or language identification. The cached outputs and verdicts of notebook 22 are re-scored under that rule, the published numbers are reproduced first as a check, and intervals that resample instruction strings (the 91 injections carry 39 distinct strings) are reported beside the record-level Wilson intervals. Host retention of each detector's selected rewrites is reported, with the manufactured-miss rate restricted to rewrites that keep at least half of the host. Scams need the judge under the final rule (markers and the judge); the judge was not run on marker-positive scam outputs in notebook 22, so scams stay on markers here and the notebook says so.
- **B. PIGuard and the conventional fine-tune on the cached 455 variants.** Static and adaptive miss rates at their document thresholds (Table 19 rows) and their columns in the transfer table (Table 22). Their own selected rewrites were not executed; the variant indices are written to disk for the GPU notebook, and the manufactured-miss rate is given on the subset of their selections that another detector also selected (already executed).
- **C. The probe at all three layers, and the cross-tab split.** The 3B features cached by notebook 52 are reloaded at layers 10, 18 and 26, the probe refitted at each, and every row of Table 27 reported per layer. The cross-tab is then split into originals and rewrites (rewrites deduplicated by variant), reported on the judge-decided categories as in the paper and on all seven categories under the final rule.
- **D. A length-only score on the second source.** Character and word counts as scores, AUROC and the miss rate at the one percent threshold, overall and by strategy.
- **E. AgentDojo: duplicate records, the tied top scores, and the 66 excluded records.** Distinct texts among the 129 benign and 1,452 attacked records; for each cached scorer, whether the benigns tied at its top score are identical texts; Table 16 recomputed on distinct benigns; and every miss rate recomputed with the 66 records excluded by the window rule counted as misses.
- **F. Section 5.2 on deepset alone.** The interval width against calibration size, subsampled within the 399 deepset benigns and within deepset plus the jailbreak benigns, for the three released guards.
- **G. Interval ends and rank severity for every detector on both document sets; the two structural ablations audited by the final rule.**
- **H. Table 15 against Table 9.** The cached scores behind the two tables compared on the records both cover, and the texts compared.
- **I. Section 6.2 by ranks.** The structure-versus-content experiment of notebook 08 re-read from its cached scores as AUROCs and paired comparisons, which recalibration cannot move, instead of raw means.
- **J. Figure 8 regenerated** from its two report files with the legend term the paper uses (manufactured miss), and a summary.

**Outputs.** `reports/part1_fixes/*.csv`, `reports/part1_fixes/summary.txt`, `figures/phase5v2.png` (regenerated), `figures/calibration_width_vs_n_deepset.png`, `data/phase5v2/score_piguard.npy`, `data/phase5v2/score_conventional_ft.npy`, `data/phase5v2/selected_extra.json`.

In [ ]:
# === SESSION BOOTSTRAP (run first): no GPU needed ===
!pip install -q transformers datasets sentencepiece langdetect scikit-learn
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
OUT = 'reports/part1_fixes'
for d_ in (OUT, 'figures', 'data/phase5v2'): os.makedirs(d_, exist_ok=True)
import torch
print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (not needed)')

## Stage A. Section 8 under the paper's final rule (Tables 20 to 22), with string-level intervals and host retention

In [ ]:
import importlib, json, hashlib, collections, numpy as np, pandas as pd
import data_loaders, phase5_redo, final_scoring, adaptive_strict, attack_success, attack_audit, strict_threshold, tie_interval, ci_tools, interval_panel, taskdrift
for m_ in (data_loaders, phase5_redo, final_scoring, adaptive_strict, attack_success, attack_audit, strict_threshold, tie_interval, ci_tools, interval_panel, taskdrift): importlib.reload(m_)
from data_loaders import load_deepset, load_bipia_categorized, load_notinject, load_jailbreak, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from phase5_redo import flat_variants, group_min_index
from final_scoring import langdetect_non_english, kappa, CATEGORIES, JUDGE_ONLY, EMOJI, BASE64, TRANSLATION, SCAMS
from adaptive_strict import strict_emoji, strict_base64, scam_markers
from attack_success import programmatic_success, combine
from attack_audit import host_retention
from strict_threshold import strict_threshold as guar, loo_fpr, fnr
from tie_interval import convention_interval, fnr_low, fnr_high, fnr_at
from ci_tools import wilson, fmt, boot_auroc
from sklearn.metrics import roc_auc_score
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
SUMMARY = []
def check(ok, *msg):
    """A reproduction check against a report already in the repository; on a dry run (NB52B_DRY set) it only prints."""
    if not ok and os.environ.get('NB52B_DRY'): print('DRY RUN, check skipped:', *msg); return
    assert ok, msg
def note(*a):
    s = ' '.join(str(x) for x in a); SUMMARY.append(s); print(s)
# the evaluation frames every stage shares
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); HOST = bip.label.values == 0; HIJ = bip.meta.isin(BIPIA_HIJACK).values; HARM = bip.meta.isin(BIPIA_HARMFUL).values
deepset = load_deepset(); yd = deepset.label.values
OPI = json.load(open('data/opi/opi_eval.json'))['records']; OY = np.array([r['label'] for r in OPI])
assert HOST.sum() == 778 and len(OPI) == 1800, (HOST.sum(), len(OPI))
# Section 8's cached run (notebook 22): the subset, the rewrites, every detector's scores, the executed outputs and the judge's answers
SUB = pd.DataFrame(json.load(open('data/phase5v2/subset.json'))); REW = json.load(open('data/phase5v2/rewrites.json')); VAR, OWNER, FIRST = flat_variants(SUB, REW)
assert len(SUB) == 91 and len(VAR) == 455 and set(SUB.meta) == set(CATEGORIES)
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)', 'full': 'structural (full)', 'full_aug': 'structural (full_aug)'}
DETS = list(LABEL); SC = {t: np.load(f'data/phase5v2/score_{t}.npy') for t in DETS}; assert all(len(SC[t]) == 455 for t in DETS)
BIPIA_SCORES = {'protectai_v2': 'data/score_protectai_v2_bipia_cat.npy', 'prompt_guard_2': 'data/score_prompt_guard_2_bipia_cat.npy', 'prompt_guard_2_22m': 'data/score_prompt_guard_2_22m_bipia_cat.npy',
                'full': 'data/structural/scores/full_bipia.npy', 'full_aug': 'data/structural/scores/full_aug_bipia.npy'}
TDOC = {t: guar(np.load(BIPIA_SCORES[t])[HOST])[0] for t in DETS}      # the paper's final threshold rule on the 778 hosts
EVA_OLD = pd.read_csv('reports/phase5v2_evasion.csv'); EVA_OLD = EVA_OLD[EVA_OLD.threshold == 'doc'].set_index('detector')
for t in DETS: check(abs(EVA_OLD.loc[LABEL[t], 't'] - TDOC[t]) <= 1e-5, t, EVA_OLD.loc[LABEL[t], 't'], TDOC[t])   # the old routine and the final rule agree where a qualifying host score exists (Appendix E)
SEL = {t: group_min_index(SC[t], OWNER) for t in DETS}
OUTS = json.load(open('data/phase5v2/exec_out.json')); JV = json.load(open('data/phase5v2/judge_7b.json'))
SRC = ['orig'] + DETS; assert all(len(OUTS[s]) == 91 for s in SRC)
DOC = {'orig': [VAR[j] for j in FIRST], **{t: [VAR[j] for j in SEL[t]] for t in DETS}}   # the document each output was produced from
def verdict(rule, cat, instr, out, doc, judged):
    """rule 'published': notebook 22's rule (attack_success.programmatic_success, else the judge).
    rule 'final': the paper's final rule, except scams, which stay on markers because the judge was not run on them here."""
    if rule == 'published':
        if cat == TRANSLATION: return bool(judged)        # langdetect was not installed in notebook 22's session, so translation went to the judge
        return bool(combine(programmatic_success(cat, instr, out), judged))
    if cat == EMOJI: return strict_emoji(out, doc)
    if cat == BASE64: return strict_base64(out, doc)
    if cat == SCAMS: return scam_markers(out)
    if cat == TRANSLATION: return bool(judged) or langdetect_non_english(out)
    return bool(judged)
RULES = ('published', 'final'); WORKS = {}
for rule in RULES:
    WORKS[rule] = {s: np.array([verdict(rule, SUB.meta.iloc[i], SUB.instruction.iloc[i], OUTS[s][i], DOC[s][i], JV.get(f'{s}|{i}', False)) for i in range(91)]) for s in SRC}
MAN_OLD = pd.read_csv('reports/phase5v2_manufactured.csv'); MAN_OLD = MAN_OLD[MAN_OLD.threshold == 'doc'].set_index('detector')
for t in DETS:   # the published numbers must come back exactly under the published rule
    missed = SC[t][SEL[t]] < TDOC[t]; man = missed & WORKS['published'][t]
    check(abs(float(man.mean()) - MAN_OLD.loc[LABEL[t], 'manufactured_rate']) < 1e-9 and abs(float(WORKS['published'][t].mean()) - MAN_OLD.loc[LABEL[t], 'still_works']) < 1e-9, t, 'Table 20 not reproduced')
check(abs(float(WORKS['published']['orig'].mean()) - MAN_OLD.orig_ASR.iloc[0]) < 1e-9, 'original success rate not reproduced')
note('Stage A: the published Table 20 numbers are reproduced from the caches under the published rule; judge answers on disk:', len(JV))
# string-level bootstrap: the 91 injections carry repeated instruction strings; resample strings, keep every record of a drawn string
STR = SUB.instruction.values; USTR = sorted(set(STR)); BY = {u: np.where(STR == u)[0] for u in USTR}
def string_boot(x, B=2000, seed=0):
    rng = np.random.default_rng(seed); v = []
    for _ in range(B):
        idx = np.concatenate([BY[USTR[k]] for k in rng.integers(0, len(USTR), len(USTR))]); v.append(float(np.asarray(x, float)[idx].mean()))
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
note(f'distinct instruction strings among the 91 injections: {len(USTR)}; per category: ' + ', '.join(f'{c} {SUB[SUB.meta == c].instruction.nunique()}' for c in sorted(set(SUB.meta))))
rows = []
for s in SRC:
    r = dict(source=s if s == 'orig' else LABEL[s], n=91)
    if s != 'orig':
        missed = SC[s][SEL[s]] < TDOC[s]; r['evades'] = fmt(missed.mean(), *wilson(int(missed.sum()), 91))
        ret = np.array([host_retention(SUB.text.iloc[i], DOC[s][i]) for i in range(91)]); r['host_half_kept'] = fmt((ret >= 0.5).mean(), *wilson(int((ret >= 0.5).sum()), 91)); r['host_retention_median'] = round(float(np.median(ret)), 3)
    for rule in RULES:
        w = WORKS[rule][s]; r[f'works_{rule}'] = fmt(w.mean(), *wilson(int(w.sum()), 91)); r[f'works_{rule}_string_boot'] = '[%.3f, %.3f]' % string_boot(w)
        if s != 'orig':
            man = missed & w; r[f'manufactured_{rule}'] = fmt(man.mean(), *wilson(int(man.sum()), 91)); r[f'manufactured_{rule}_n'] = int(man.sum()); r[f'manufactured_{rule}_string_boot'] = '[%.3f, %.3f]' % string_boot(man)
            k = ret >= 0.5; r[f'manufactured_{rule}_among_host_half_kept'] = fmt(man[k].mean(), *wilson(int(man[k].sum()), int(k.sum()))) if k.sum() else 'n/a'
    rows.append(r)
A1 = pd.DataFrame(rows); A1.to_csv(f'{OUT}/phase5v2_rules.csv', index=False); print('\n=== Table 20 under the published rule and the final rule (document thresholds), with host retention ==='); print(A1.to_string(index=False))
crow = []
for cat in sorted(set(SUB.meta)):
    m = (SUB.meta == cat).values; r = dict(category=cat, n=int(m.sum()), strings=int(SUB[m].instruction.nunique()))
    for rule in RULES:
        r[f'orig_ASR_{rule}'] = round(float(WORKS[rule]['orig'][m].mean()), 3)
        for t in DETS: r[f'manufactured_{rule}_{t}'] = round(float(((SC[t][SEL[t]] < TDOC[t]) & WORKS[rule][t])[m].mean()), 3)
    crow.append(r)
A2 = pd.DataFrame(crow); A2.to_csv(f'{OUT}/phase5v2_by_category_rules.csv', index=False); print('\n=== Table 21 under both rules ==='); print(A2.to_string(index=False))
trow = []
for rule in RULES:
    for t in DETS:
        man = (SC[t][SEL[t]] < TDOC[t]) & WORKS[rule][t]; r = dict(rule=rule, source_detector=LABEL[t], n_manufactured=int(man.sum()))
        for d in DETS: r[f'caught_by_{d}'] = round(float((SC[d][SEL[t]] >= TDOC[d])[man].mean()), 3) if man.sum() else np.nan
        trow.append(r)
A3 = pd.DataFrame(trow); A3.to_csv(f'{OUT}/phase5v2_transfer_rules.csv', index=False); print('\n=== Table 22 under both rules (the diagonal is zero by construction) ==='); print(A3.to_string(index=False))
chg = {t: int((WORKS['published'][t] != WORKS['final'][t]).sum()) for t in SRC}
note('Stage A: verdicts that change between the published rule and the final rule, per source:', chg, '| the final rule here keeps scams on markers (the judge was not run on marker-positive scam outputs in notebook 22)')

## Stage B. PIGuard and the conventional fine-tune on the 455 cached variants (Tables 19 and 22; Table 20 where already executed)

In [ ]:
from detectors import score_released
from structural_train import score_texts
PAN = pd.read_csv('reports/guard_panel_extended_ci.csv').set_index('model')
EXTRA = {'piguard': ('PIGuard', 'data/closers/score_piguard_bipia.npy'), 'conventional_ft': ('conventional fine-tune', 'data/structural/scores/conventional_ft_bipia.npy')}
# notebook 24 flips a guard's scores when its direct-set AUROC is below 0.5 (the loader guessed the label order); the same test decides here
INV = {'piguard': bool(roc_auc_score(yd, np.load('data/closers/score_piguard_deepset.npy')) < 0.5), 'conventional_ft': False}
for t, (nm, _) in EXTRA.items():
    fp = f'data/phase5v2/score_{t}.npy'
    if os.path.exists(fp): print(nm, 'scores on the 455 variants loaded from cache'); continue
    if t == 'piguard':
        cid = json.load(open('data/closers/id_piguard.json')); print('scoring the 455 variants with', cid); p = score_released(VAR, cid)
        if INV['piguard']: p = 1.0 - p; print('PIGuard: scores inverted, as in notebook 24')
    else:
        mdir = 'data/structural/model_conventional_ft/best'; assert os.path.exists(os.path.join(mdir, 'config.json')), mdir; print('scoring the 455 variants with the conventional fine-tune'); p = score_texts(mdir, VAR)
    np.save(fp, np.asarray(p))
for t, (nm, fb) in EXTRA.items():
    SC[t] = np.load(f'data/phase5v2/score_{t}.npy'); assert len(SC[t]) == 455; LABEL[t] = nm
    pb = np.load(fb)
    if INV[t]: pb = 1.0 - pb       # the caches hold raw scores; notebook 24 flipped them in memory, and the variant scores above were flipped the same way
    TDOC[t] = guar(pb[HOST])[0]; SEL[t] = group_min_index(SC[t], OWNER)
    check(abs(fnr(pb[HIJ], TDOC[t]) - float(str(PAN.loc[nm, 'FNR_hijack_doc']).split()[0])) < 0.02, nm, 'host threshold does not reproduce Table 9')
XD = list(EXTRA); rows = []
for t in XD:
    ps, pa = SC[t][FIRST], SC[t][SEL[t]]; ms, ma = ps < TDOC[t], pa < TDOC[t]
    rows.append(dict(detector=LABEL[t], threshold_doc=TDOC[t], mean_score_originals=round(float(ps.mean()), 4), mean_score_selected=round(float(pa.mean()), 4),
                     FNR_static=fmt(ms.mean(), *wilson(int(ms.sum()), 91)), FNR_adaptive=fmt(ma.mean(), *wilson(int(ma.sum()), 91))))
B1 = pd.DataFrame(rows); B1.to_csv(f'{OUT}/phase5v2_extra_evasion.csv', index=False); print('=== Table 19 rows for PIGuard and the conventional fine-tune (document thresholds, attacker second) ==='); print(B1.to_string(index=False))
# their columns in Table 22: the five detectors' manufactured misses (both rules) scored by the two at their own document thresholds
trow = []
for rule in RULES:
    for t in DETS:
        man = (SC[t][SEL[t]] < TDOC[t]) & WORKS[rule][t]; r = dict(rule=rule, source_detector=LABEL[t], n_manufactured=int(man.sum()))
        for d in XD: r[f'caught_by_{d}'] = round(float((SC[d][SEL[t]] >= TDOC[d])[man].mean()), 3) if man.sum() else np.nan
        trow.append(r)
B2 = pd.DataFrame(trow); B2.to_csv(f'{OUT}/phase5v2_extra_transfer.csv', index=False); print('\n=== Table 22, two new columns ==='); print(B2.to_string(index=False))
# their own selected rewrites: executed only where another detector selected the same variant
rows = []; SELX = {}
for t in XD:
    sel = SEL[t]; cov = []
    for i in range(91):
        src = next((d for d in DETS if SEL[d][i] == sel[i]), 'orig' if sel[i] == FIRST[i] else None); cov.append(src)
    have = np.array([c is not None for c in cov]); missed = SC[t][sel] < TDOC[t]
    for rule in RULES:
        w = np.array([bool(WORKS[rule][c][i]) if c is not None else False for i, c in enumerate(cov)]); man = missed & w & have
        rows.append(dict(detector=LABEL[t], rule=rule, selected_already_executed=int(have.sum()), of=91, evades_among_executed=fmt(missed[have].mean(), *wilson(int(missed[have].sum()), int(have.sum()))) if have.sum() else 'n/a',
                         manufactured_among_executed=fmt(man[have].mean(), *wilson(int(man[have].sum()), int(have.sum()))) if have.sum() else 'n/a', not_executed=int((~have).sum())))
    SELX[t] = dict(selected_variant_index=[int(x) for x in sel], already_executed_source=cov, threshold_doc=float(TDOC[t]))
json.dump(SELX, open('data/phase5v2/selected_extra.json', 'w'))
B3 = pd.DataFrame(rows); B3.to_csv(f'{OUT}/phase5v2_extra_manufactured_partial.csv', index=False); print('\n=== Table 20 for the two, on the selected rewrites that are already executed (the rest need the GPU notebook) ==='); print(B3.to_string(index=False))
note('Stage B:', ', '.join(f"{LABEL[t]} static {B1[B1.detector == LABEL[t]].FNR_static.iloc[0]} adaptive {B1[B1.detector == LABEL[t]].FNR_adaptive.iloc[0]}" for t in XD), '| selected rewrites not yet executed:', {LABEL[t]: int(B3[B3.detector == LABEL[t]].not_executed.iloc[0]) for t in XD}, '(indices in data/phase5v2/selected_extra.json)')

## Stage C. The 3B probe at layers 10, 18 and 26, and the cross-tab split into originals and rewrites

In [ ]:
from taskdrift import fit_probe, probe_scores
from structural_train import load_jsonl
# notebook 52's draws, in notebook 52's order, so the cached feature rows are the same records (its cell 3, verbatim where it matters)
rng = np.random.default_rng(0)
def probe_set(path, n_pos, n_neg):
    R = [r for r in load_jsonl(path) if r['form'] != 'standalone']; pos = [r for r in R if r['label'] == 1]; neg = [r for r in R if r['label'] == 0]
    pos = [pos[i] for i in rng.choice(len(pos), min(n_pos, len(pos)), replace=False)]; neg = [neg[i] for i in rng.choice(len(neg), min(n_neg, len(neg)), replace=False)]
    return [r['text'] for r in pos + neg], np.array([1] * len(pos) + [0] * len(neg))
TR_X, TR_Y = probe_set('data/structural/train.jsonl', 600, 600); VA_X, VA_Y = probe_set('data/structural/val.jsonl', 150, 150)
oi = rng.choice(len(OPI), 600, replace=False); OPI_Y6 = np.array([OPI[i]['label'] for i in oi])
noti = load_notinject(); HOW = json.load(open('data/closers/howto.json'))['texts'][:200]
AD = json.load(open('data/agentdojo/set.json')); AD_b = [r for r in AD if r['label'] == 0]; AD_a = [r for r in AD if r['label'] == 1 and r.get('inj_offset', 0) <= 1600]; AD_a = [AD_a[i] for i in rng.choice(len(AD_a), 400, replace=False)]
AD_Y = np.array([0] * len(AD_b) + [1] * len(AD_a))
SIZES = {'train': len(TR_X), 'val': len(VA_X), 'bipia': len(bip), 'opi': 600, 'notinject': len(noti), 'howto': len(HOW), 'agentdojo': len(AD_Y), 'adv': 455}
LAYERS = [10, 18, 26]; FEAT = {}
for s_, n_ in SIZES.items():
    FEAT[s_] = {L: np.load(f'data/taskdrift_3b/feat_summarize_{s_}_L{L}.npy') for L in LAYERS}
    assert all(FEAT[s_][L].shape[0] == n_ for L in LAYERS), (s_, n_, {L: FEAT[s_][L].shape for L in LAYERS})
yh = np.r_[np.ones(int(HIJ.sum())), np.zeros(int(HOST.sum()))]; yr = np.r_[np.ones(int(HARM.sum())), np.zeros(int(HOST.sum()))]
SEL_FULL = SEL['full']; LAYROWS, PS = [], {}
for L in LAYERS:
    P = fit_probe(FEAT['train'][L], TR_Y, FEAT['val'][L], VA_Y); S = {s_: probe_scores(P, FEAT[s_][L]) for s_ in SIZES}; PS[L] = S
    hosts = S['bipia'][HOST]; tg, nb_, _ = guar(hosts); to, _, _ = guar(S['opi'][OPI_Y6 == 0]); ta, _, _ = guar(S['agentdojo'][AD_Y == 0]); A = S['adv']; sel2 = group_min_index(A, OWNER)
    r = dict(layer=L, C=P['C'], val_AUROC=round(P['val_auroc'], 4), hosts_blocked=nb_,
             hijack_AUROC=fmt(roc_auc_score(yh, np.r_[S['bipia'][HIJ], hosts]), *boot_auroc(yh, np.r_[S['bipia'][HIJ], hosts])), hijack_FNR=fmt(fnr(S['bipia'][HIJ], tg), *wilson(int((S['bipia'][HIJ] < tg).sum()), int(HIJ.sum()))),
             harmful_AUROC=fmt(roc_auc_score(yr, np.r_[S['bipia'][HARM], hosts]), *boot_auroc(yr, np.r_[S['bipia'][HARM], hosts])), harmful_FNR=fmt(fnr(S['bipia'][HARM], tg), *wilson(int((S['bipia'][HARM] < tg).sum()), int(HARM.sum()))),
             second_source_AUROC=round(roc_auc_score(OPI_Y6, S['opi']), 3), second_source_FNR=round(fnr(S['opi'][OPI_Y6 == 1], to), 3),
             agentdojo_AUROC=fmt(roc_auc_score(AD_Y, S['agentdojo']), *boot_auroc(AD_Y, S['agentdojo'])), agentdojo_FNR=round(fnr(S['agentdojo'][AD_Y == 1], ta), 3),
             notinject_FPR=round(float((S['notinject'] >= tg).mean()), 3), recipes_FPR=round(float((S['howto'] >= tg).mean()), 3),
             adv_static_FNR=fmt(fnr(A[FIRST], tg), *wilson(int((A[FIRST] < tg).sum()), 91)), adv_first_FNR=fmt(fnr(A[SEL_FULL], tg), *wilson(int((A[SEL_FULL] < tg).sum()), 91)), adv_second_FNR=fmt(fnr(A[sel2], tg), *wilson(int((A[sel2] < tg).sum()), 91)))
    LAYROWS.append(r)
C1 = pd.DataFrame(LAYROWS); C1.to_csv(f'{OUT}/taskdrift3b_all_layers.csv', index=False); print('=== the 3B probe at each of the three layers (Table 27 rows; the paper reports layer 18) ==='); print(C1.to_string(index=False))
OLD = pd.read_csv('reports/taskdrift3b_eval.csv'); OLD = OLD[OLD.model.str.startswith('Qwen2.5-3B')]
old_h = float(OLD[OLD.distribution == 'BIPIA hijack vs hosts'].AUROC.iloc[0].split()[0]); new_h = float(C1[C1.layer == 18].hijack_AUROC.iloc[0].split()[0])
check(abs(old_h - new_h) < 0.005, 'layer 18 does not reproduce notebook 52', old_h, new_h)
note('Stage C: hijack AUROC at layer 18, notebook 52 against this refit:', old_h, new_h, '; hijack AUROC by layer:', {int(r.layer): r.hijack_AUROC.split()[0] for r in C1.itertuples()}, '; attacker-second miss rate by layer:', {int(r.layer): r.adv_second_FNR.split()[0] for r in C1.itertuples()})
# the cross-tab, split. Items: the 91 originals and every selected rewrite of the five detectors; a rewrite selected by several detectors is one item
def newcombe(k1, n1, k0, n0):
    p1, p0 = k1 / n1, k0 / n0; lo1, hi1 = wilson(k1, n1); lo0, hi0 = wilson(k0, n0)
    return p1 - p0, (p1 - p0) - np.sqrt((p1 - lo1) ** 2 + (hi0 - p0) ** 2), (p1 - p0) + np.sqrt((hi1 - p1) ** 2 + (p0 - lo0) ** 2)
def items(rule, cats, dedup):
    """(variant index, category, works) for originals and for the five detectors' selected rewrites; dedup keeps one row per variant."""
    out, seen = [], set()
    for s in SRC:
        for i in range(91):
            if SUB.meta.iloc[i] not in cats: continue
            if rule == 'judge' and f'{s}|{i}' not in JV: continue
            v = int(FIRST[i] if s == 'orig' else SEL[s][i])
            if dedup and v in seen: continue
            seen.add(v); w = bool(JV[f'{s}|{i}']) if rule == 'judge' else bool(WORKS[rule][s][i]); out.append((v, SUB.meta.iloc[i], s == 'orig', w))
    return out
rows = []
for L in LAYERS:
    A = PS[L]['adv']; tg = guar(PS[L]['bipia'][HOST])[0]; flag = A >= tg
    for rule, cats, lab in (('judge', set(JUDGE_ONLY), 'judge-decided categories, as the paper'), ('final', set(CATEGORIES), 'all seven categories, final rule'), ('final', set(CATEGORIES) - set(JUDGE_ONLY), 'marker categories only, final rule')):
        for dedup in (False, True):
            IT = items(rule, cats, dedup)
            for part, keep in (('all', lambda o: True), ('originals', lambda o: o), ('rewrites', lambda o: not o)):
                sub = [(v, w) for v, c, o, w in IT if keep(o)]
                if not sub: continue
                f1 = [flag[v] for v, w in sub if w]; f0 = [flag[v] for v, w in sub if not w]
                if not f1 or not f0: rows.append(dict(layer=L, items=lab, rewrites_deduplicated=dedup, part=part, n_followed=len(f1), n_not_followed=len(f0))); continue
                d, lo, hi = newcombe(int(sum(f1)), len(f1), int(sum(f0)), len(f0))
                rows.append(dict(layer=L, items=lab, rewrites_deduplicated=dedup, part=part, n_followed=len(f1), followed_flagged=fmt(np.mean(f1), *wilson(int(sum(f1)), len(f1))), n_not_followed=len(f0), not_followed_flagged=fmt(np.mean(f0), *wilson(int(sum(f0)), len(f0))),
                                 difference=round(d, 3), difference_lo=round(float(lo), 3), difference_hi=round(float(hi), 3)))
C2 = pd.DataFrame(rows); C2.to_csv(f'{OUT}/taskdrift3b_crosstab_split.csv', index=False)
chk = C2[(C2.layer == 18) & (C2['items'].str.startswith('judge')) & (~C2.rewrites_deduplicated) & (C2.part == 'all')].iloc[0]
old = pd.read_csv('reports/taskdrift3b_crosstab.csv').iloc[0]; check(int(chk.n_followed) == int(old.followed_n) and int(chk.n_not_followed) == int(old.not_followed_n), 'the paper\'s cross-tab is not reproduced', int(chk.n_followed), int(old.followed_n))
print('\n=== the cross-tab split into originals and rewrites, by layer, item set and deduplication (Newcombe interval on the difference) ==='); print(C2[C2.layer == 18].to_string(index=False)); print('\n(other layers in the file)')
r18 = C2[(C2.layer == 18) & C2['items'].str.startswith('all') & C2.rewrites_deduplicated]
note('Stage C: layer 18, all seven categories under the final rule, rewrites deduplicated: ' + '; '.join(f"{r.part}: followed {r.n_followed} flagged {getattr(r, 'followed_flagged', 'n/a')}, not followed {r.n_not_followed} flagged {getattr(r, 'not_followed_flagged', 'n/a')}, difference {getattr(r, 'difference', 'n/a')} [{getattr(r, 'difference_lo', '')}, {getattr(r, 'difference_hi', '')}]" for r in r18.itertuples()))

## Stage D. The second source scored by length alone

In [ ]:
TXT = [r['text'] for r in OPI]; STRAT = np.array([r.get('strategy', '') for r in OPI]); TASK = np.array([r.get('target_task', '') for r in OPI])
LEN = {'characters': np.array([len(t) for t in TXT], float), 'words': np.array([len(t.split()) for t in TXT], float)}
rows = []
for nm, x in LEN.items():
    t1, k1, f1 = guar(x[OY == 0]); r = dict(score=nm, AUROC=fmt(roc_auc_score(OY, x), *boot_auroc(OY, x)), FNR_at_1pct=fmt(fnr(x[OY == 1], t1), *wilson(int((x[OY == 1] < t1).sum()), int((OY == 1).sum()))), threshold=t1, benigns_blocked=k1)
    for s_ in sorted(set(STRAT) - {''}):
        m = (OY == 1) & (STRAT == s_); r[f'AUROC_{s_}'] = round(roc_auc_score(np.r_[np.ones(m.sum()), np.zeros((OY == 0).sum())], np.r_[x[m], x[OY == 0]]), 3)
    rows.append(r)
# every injected record is its host plus the injected instruction and data, so the pair is longer by construction
pair_longer = np.mean([len(OPI[i + 1]['text']) > len(OPI[i]['text']) for i in range(0, len(OPI) - 1, 2) if OPI[i]['label'] == 0 and OPI[i + 1]['label'] == 1])
D1 = pd.DataFrame(rows); D1.to_csv(f'{OUT}/opi_length_only.csv', index=False); print('=== the second source scored by length alone (Section 10.3 said this was not computed) ==='); print(D1.to_string(index=False))
LEX = pd.read_csv('reports/opi_eval.csv'); lex = float(LEX[LEX.model == 'TF-IDF reference'].AUROC.iloc[0])
note(f'Stage D: length alone separates the second source at AUROC {D1.AUROC.iloc[0].split()[0]} (characters) and {D1.AUROC.iloc[1].split()[0]} (words); the lexical reference is {lex:.3f}; injected record longer than its host in {pair_longer:.3f} of pairs')

## Stage E. AgentDojo: duplicate records, tied top scores, and the 66 records the window rule excluded

In [ ]:
R_ALL = AD; WINDOW_CHARS = 1600
KEEP = np.array([i for i, r in enumerate(R_ALL) if r['label'] == 0 or r.get('inj_offset', 0) <= WINDOW_CHARS]); EXCL = [r for r in R_ALL if r['label'] == 1 and r.get('inj_offset', 0) > WINDOW_CHARS]
R = [R_ALL[i] for i in KEEP]; Y = np.array([r['label'] for r in R]); ben, atk = Y == 0, Y == 1; TXTA = [r['text'] for r in R]
note(f'Stage E: AgentDojo records after the window rule: {int(ben.sum())} benign, {int(atk.sum())} attacked; excluded by the window rule: {len(EXCL)}')
assert int(ben.sum()) == 129 and int(atk.sum()) == 1452 and len(EXCL) == 66, (ben.sum(), atk.sum(), len(EXCL))
ben_txt = [TXTA[i] for i in np.where(ben)[0]]; atk_txt = [TXTA[i] for i in np.where(atk)[0]]
cb = collections.Counter(ben_txt); ca = collections.Counter(atk_txt)
dup_b = {t: n for t, n in cb.items() if n > 1}; dup_a = {t: n for t, n in ca.items() if n > 1}
note(f'distinct benign texts {len(cb)} of 129 ({sum(dup_b.values()) - len(dup_b)} duplicate copies in {len(dup_b)} groups); distinct attacked texts {len(ca)} of 1452 ({sum(dup_a.values()) - len(dup_a)} duplicate copies in {len(dup_a)} groups)')
LAB_AD = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)', 'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert',
          'full': 'structural (full)', 'full_aug': 'structural (full_aug)', 'conventional_ft': 'conventional fine-tune', 'piguard': 'PIGuard'}
SA = {}
for t, nm in LAB_AD.items():
    fp = f'data/agentdojo/score_{t}.npy'
    if os.path.exists(fp): SA[nm] = np.load(fp)[KEEP]
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
STR_TRAIN = load_jsonl('data/structural/train.jsonl'); vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
lr = LogisticRegression(C=1.0, max_iter=2000).fit(vec.fit_transform([r['text'] for r in STR_TRAIN]), [r['label'] for r in STR_TRAIN]); SA['TF-IDF reference'] = lr.predict_proba(vec.transform(TXTA))[:, 1]   # as notebook 47
OLDA = pd.read_csv('reports/agentdojo_eval_strict.csv').set_index('model')
# one row per benign text (first occurrence), one per attacked text
first_b = sorted({t: i for i, t in reversed(list(enumerate(ben_txt)))}.values()); first_a = sorted({t: i for i, t in reversed(list(enumerate(atk_txt)))}.values())
bidx, aidx = np.where(ben)[0][first_b], np.where(atk)[0][first_a]
rows = []
for nm, p in SA.items():
    pb, pa = p[ben], p[atk]; top = pb.max(); tied = np.where(pb == top)[0]; tied_txt = [ben_txt[i] for i in tied]
    ts, k, f = guar(pb); pbd, pad = p[bidx], p[aidx]; tsd, kd, fd = guar(pbd); misses_full = int((pa < ts).sum())
    rows.append(dict(model=nm, AUROC=round(roc_auc_score(Y, p), 3), benigns_tied_at_top=len(tied), tied_benigns_distinct_texts=len(set(tied_txt)), blocked_strict=k, loo_FPR=round(loo_fpr(pb), 4),
                     FNR_strict=fmt(misses_full / len(pa), *wilson(misses_full, len(pa))), FNR_strict_reported=OLDA.FNR_strict.get(nm, np.nan),
                     n_benign_distinct=len(pbd), n_attacked_distinct=len(pad), AUROC_distinct=round(roc_auc_score(np.r_[np.ones(len(pad)), np.zeros(len(pbd))], np.r_[pad, pbd]), 3), blocked_strict_distinct=kd, loo_FPR_distinct=round(loo_fpr(pbd), 4),
                     FNR_strict_distinct=fmt(fnr(pad, tsd), *wilson(int((pad < tsd).sum()), len(pad))),
                     FNR_with_excluded_as_misses=fmt((misses_full + len(EXCL)) / (len(pa) + len(EXCL)), *wilson(misses_full + len(EXCL), len(pa) + len(EXCL)))))
E1 = pd.DataFrame(rows); E1.to_csv(f'{OUT}/agentdojo_dedup_and_excluded.csv', index=False)
print('=== AgentDojo: tied top benign scores and whether they are duplicate texts; Table 16 on distinct records; miss rates with the 66 excluded records counted as misses ==='); print(E1.to_string(index=False))
for nm in E1.model: check(abs(float(E1[E1.model == nm].FNR_strict.iloc[0].split()[0]) - float(OLDA.FNR_strict.get(nm, np.nan))) < (0.02 if nm == 'TF-IDF reference' else 0.0015), nm, 'does not reproduce reports/agentdojo_eval_strict.csv')
note('Stage E: tied top benigns (count, distinct texts): ' + ', '.join(f'{r.model} {r.benigns_tied_at_top}/{r.tied_benigns_distinct_texts}' for r in E1.itertuples() if r.benigns_tied_at_top > 1) + '; miss rate with the excluded records counted: ' + ', '.join(f'{r.model} {r.FNR_with_excluded_as_misses.split()[0]}' for r in E1.itertuples()))

## Stage F. Section 5.2 on deepset alone: interval width against calibration size within the source that opened it

In [ ]:
jb = load_jailbreak(); yj = jb.label.values
TAGS3 = ['protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m']
P3 = {t: {k: np.load(f'data/score_{t}_{k}.npy') for k in ('deepset', 'bipia_cat', 'jailbreak')} for t in TAGS3}
for t in TAGS3: assert len(P3[t]['deepset']) == len(deepset) and len(P3[t]['bipia_cat']) == len(bip) and len(P3[t]['jailbreak']) == len(jb)
ATK3 = {t: {'direct': P3[t]['deepset'][yd == 1], 'indirect_harmful': P3[t]['bipia_cat'][HARM], 'indirect_hijack': P3[t]['bipia_cat'][HIJ], 'jailbreak': P3[t]['jailbreak'][yj == 1]} for t in TAGS3}
POOLS = {'deepset (399)': lambda t: P3[t]['deepset'][yd == 0], 'deepset + jailbreak benigns': lambda t: np.concatenate([P3[t]['deepset'][yd == 0], P3[t]['jailbreak'][yj == 0]])}
def widths(sub, atk):
    iv = convention_interval(sub); tg, kg, fg = guar(sub); r = dict(N=iv['n'], blocked_convention=iv['k'], blocked_guar=kg)
    for nm, pa in atk.items(): r[f'width_{nm}'] = fnr_high(pa, iv) - fnr_low(pa, iv); r[f'FNR_guar_{nm}'] = fnr_at(pa, tg)
    return r
rows = []
for t in TAGS3:
    for pname, fn in POOLS.items():
        pool = fn(t); g = np.random.default_rng(0)
        for N0 in [50, 100, 200, 300, 399, 600, 797, 1000]:
            if N0 > len(pool): continue
            for dr in range(20):
                sub = pool[g.choice(len(pool), N0, replace=False)]; rows.append(dict(detector=LABEL[t], pool=pname, draw=dr, **widths(sub, ATK3[t])))
F1 = pd.DataFrame(rows); F1.to_csv(f'{OUT}/calibration_width_vs_n_deepset.csv', index=False)
F2 = F1.groupby(['detector', 'pool', 'N']).agg(width_hijack=('width_indirect_hijack', 'mean'), width_hijack_p5=('width_indirect_hijack', lambda v: np.percentile(v, 5)), width_hijack_p95=('width_indirect_hijack', lambda v: np.percentile(v, 95)),
                                                width_harmful=('width_indirect_harmful', 'mean'), width_direct=('width_direct', 'mean'), FNR_guar_hijack=('FNR_guar_indirect_hijack', 'mean')).reset_index()
F2.to_csv(f'{OUT}/calibration_width_vs_n_deepset_summary.csv', index=False); print('=== Section 5.2 on the source that opened the interval: width against N within deepset, and within deepset plus the jailbreak benigns (20 draws) ==='); print(F2.round(3).to_string(index=False))
POOLED = pd.read_csv('reports/calibration_width_vs_n_v2.csv'); pooled400 = POOLED[POOLED.N == 400].groupby('detector').width_indirect_hijack.mean()
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, pname in zip(axes, POOLS):
    for t in TAGS3:
        d_ = F2[(F2.detector == LABEL[t]) & (F2.pool == pname)]; ax.plot(d_.N, d_.width_hijack, marker='o', label=LABEL[t]); ax.fill_between(d_.N, d_.width_hijack_p5, d_.width_hijack_p95, alpha=.15)
    ax.axvline(399, ls=':', color='k', lw=.8); ax.set_xscale('log'); ax.set_xlabel('calibration benigns N'); ax.set_title(f'hijack interval width, subsampled within {pname}'); ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[0].set_ylabel('identification interval width'); plt.tight_layout(); plt.savefig('figures/calibration_width_vs_n_deepset.png', dpi=150, bbox_inches='tight'); plt.show()
d399 = F2[(F2.pool == 'deepset (399)') & (F2.N == 399)].set_index('detector').width_hijack; d797 = F2[(F2.pool == 'deepset + jailbreak benigns') & (F2.N == 797)].set_index('detector').width_hijack if (F2.N == 797).any() else None
note('Stage F: hijack width at N = 399 within deepset: ' + ', '.join(f'{k} {v:.3f}' for k, v in d399.items()) + (' | at N = 797 within deepset plus jailbreak benigns: ' + ', '.join(f'{k} {v:.3f}' for k, v in d797.items()) if d797 is not None else '') + ' | pooled four-source mixture at N = 400 (Figure 2): ' + ', '.join(f'{k} {v:.3f}' for k, v in pooled400.items()))

## Stage G. Interval ends and rank severity for every detector on both document sets; the two structural ablations under the final rule

In [ ]:
# every detector with cached scores on the BIPIA documents and the second source (the loading of notebook 24, cell 10)
KEY = {'bipia': 'bipia_cat', 'notinject': 'notinject', 'deepset': 'deepset', 'jailbreak': 'jailbreak'}
REL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
EXT = {'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert', 'piguard': 'PIGuard'}
SALL, SRCF = {}, {}
for t, nm in REL.items(): SALL[nm] = {'bipia': np.load(f'data/score_{t}_bipia_cat.npy'), 'deepset': np.load(f'data/score_{t}_deepset.npy'), 'opi': np.load(f'data/opi/score_{t}.npy')}; SRCF[nm] = 'data/score_*'
for t, nm in EXT.items():
    fps = {s: f'data/closers/score_{t}_{s}.npy' for s in ('bipia', 'deepset', 'opi')}
    if all(os.path.exists(f) for f in fps.values()): SALL[nm] = {s: np.load(f) for s, f in fps.items()}; SRCF[nm] = 'data/closers/score_*'
for cfg, nm in (('full', 'structural (full)'), ('full_aug', 'structural (full_aug)'), ('ablation_no_twins', 'structural (ablation_no_twins)'), ('full_sizematched', 'structural (full_sizematched)'), ('conventional_ft', 'conventional fine-tune')):
    fps = {'bipia': f'data/structural/scores/{cfg}_bipia.npy', 'deepset': f'data/structural/scores/{cfg}_deepset.npy', 'opi': f'data/opi/score_structural_{cfg}.npy' if cfg != 'conventional_ft' else 'data/structural/scores/conventional_ft_opi.npy'}
    if all(os.path.exists(f) for f in fps.values()): SALL[nm] = {s: np.load(f) for s, f in fps.items()}; SRCF[nm] = 'data/structural/scores/*, data/opi/*'
    else: print(nm, 'not loaded; missing', [f for f in fps.values() if not os.path.exists(f)])
for nm in list(SALL):
    if nm in ('deepset injection', 'fmops distilbert', 'ProtectAI-v1', 'PIGuard') and roc_auc_score(yd, SALL[nm]['deepset']) < 0.5: SALL[nm] = {k: 1 - v for k, v in SALL[nm].items()}; print(nm, 'inverted')
    assert len(SALL[nm]['bipia']) == len(bip) and len(SALL[nm]['opi']) == len(OPI), nm
def rank_R(p_miss, hosts): return float(np.mean([(hosts > v).mean() for v in p_miss])) if len(p_miss) else np.nan
rows = []
for nm, sc in SALL.items():
    pb, po = sc['bipia'], sc['opi']; hosts = pb[HOST]; ob = po[OY == 0]
    for setname, cal, atks in (('BIPIA hosts (778)', hosts, {'hijack': pb[HIJ], 'harmful': pb[HARM]}), ('second source benigns (900)', ob, {'injected': po[OY == 1]})):
        iv = convention_interval(cal); tg, kg, fg = guar(cal); r = dict(detector=nm, calibration=setname, blocked_convention=iv['k'], blocked_guar=kg, calFPR_guar=round(fg, 4), loo_FPR=round(loo_fpr(cal), 4), v_lo=iv['v_lo'], v_hi=iv['v_hi'], t_guar=tg)
        for an, pa in atks.items():
            r[f'FNR_low_{an}'] = round(fnr_low(pa, iv), 3); r[f'FNR_high_{an}'] = round(fnr_high(pa, iv), 3); r[f'FNR_guar_{an}'] = round(fnr_at(pa, tg), 3); r[f'width_{an}'] = round(fnr_high(pa, iv) - fnr_low(pa, iv), 3)
            r[f'R_mean_on_misses_{an}'] = round(rank_R(pa[pa < tg], cal), 3)
        rows.append(r)
G1 = pd.DataFrame(rows); G1.to_csv(f'{OUT}/document_interval_all_detectors.csv', index=False)
print('=== the identification interval ends, the guaranteed miss rate and rank severity R for every detector on both document sets ==='); print(G1.drop(columns=['v_lo', 'v_hi', 't_guar']).to_string(index=False))
wmax = G1[[c for c in G1.columns if c.startswith('width_')]].max(axis=1)
note('Stage G: largest interval width, by detector and calibration set: ' + ', '.join(f'{nm} on {cal.split(" (")[0]} {w:.3f}' for nm, cal, w in zip(G1.detector, G1.calibration, wmax)) + '; every width at or under 0.02: ' + str(bool((wmax <= 0.02).all())))
# the two structural ablations under the final threshold rule (Appendix E listed them as not audited)
rows = []
for nm in ('structural (ablation_no_twins)', 'structural (full_sizematched)'):
    if nm not in SALL: continue
    sc = SALL[nm]
    for setname, cal, pa in (('BIPIA hosts (778)', sc['bipia'][HOST], sc['bipia'][HIJ]), ('second source benigns', sc['opi'][OY == 0], sc['opi'][OY == 1])):
        told = interval_panel.three_thresholds(cal)['guar']; ts, k, f = guar(cal)
        rows.append(dict(detector=nm, calibration_set=setname, n_benign=len(cal), blocked_old=int((cal >= told).sum()), calFPR_old=round(float((cal >= told).mean()), 4), blocked_strict=k, calFPR_strict=round(f, 4), FNR_old=round(fnr(pa, told), 3), FNR_strict=round(fnr(pa, ts), 3), fallback_fired=bool(ts != told), loo_FPR_strict=round(loo_fpr(cal), 4)))
G2 = pd.DataFrame(rows); G2.to_csv(f'{OUT}/threshold_audit_ablations.csv', index=False); print('\n=== the two structural ablations audited by the final rule (same columns as reports/threshold_audit.csv) ==='); print(G2.to_string(index=False))
note('Stage G: ablation audit, fallback fired in', int(G2.fallback_fired.sum()) if len(G2) else 'n/a', 'of', len(G2), 'cells; old and strict miss rates agree:', bool((G2.FNR_old == G2.FNR_strict).all()) if len(G2) else 'n/a')

## Stage H. Table 15 against Table 9: the two cached score runs on the same records

In [ ]:
POS = json.load(open('data/closers/positions.json')); atk_idx = np.array(POS['idx']); assert len(atk_idx) == 210
same_text = np.array([POS['end'][k] == bip.text.iloc[i] for k, i in enumerate(atk_idx)])
note(f'Stage H: the "end" re-embedding reproduces the original BIPIA text in {int(same_text.sum())} of 210 records (the rest differ in whitespace or line breaks)')
rows = []
for t, nm in {**REL, **{'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert', 'piguard': 'PIGuard'}, 'full': 'structural (full)', 'full_aug': 'structural (full_aug)', 'conventional_ft': 'conventional fine-tune'}.items():
    fe = f'data/closers/score_{t}_pos_end.npy'
    if not os.path.exists(fe) or nm not in SALL: continue
    pe = np.load(fe); pe = 1 - pe if (nm in ('deepset injection', 'fmops distilbert', 'ProtectAI-v1', 'PIGuard') and roc_auc_score(yd, np.load(f'data/closers/score_{t}_deepset.npy')) < 0.5) else pe
    p9 = SALL[nm]['bipia'][atk_idx]; d = np.abs(pe - p9); hosts = SALL[nm]['bipia'][HOST]; tg = guar(hosts)[0]; hj = HIJ[atk_idx]
    rows.append(dict(detector=nm, identical_texts=int(same_text.sum()), max_abs_diff_all=float(d.max()), max_abs_diff_identical_texts=float(d[same_text].max()) if same_text.any() else np.nan, share_diff_over_1e3=round(float((d > 1e-3).mean()), 3),
                     share_diff_over_1e3_identical_texts=round(float((d[same_text] > 1e-3).mean()), 3) if same_text.any() else np.nan,
                     hijack_AUROC_table9=round(roc_auc_score(np.r_[np.ones(hj.sum()), np.zeros(HOST.sum())], np.r_[p9[hj], hosts]), 3), hijack_AUROC_table15_end=round(roc_auc_score(np.r_[np.ones(hj.sum()), np.zeros(HOST.sum())], np.r_[pe[hj], hosts]), 3),
                     hijack_FNR_table9=round(fnr(p9[hj], tg), 3), hijack_FNR_table15_end=round(fnr(pe[hj], tg), 3), hosts_rescored_in_notebook_23=False))
H1 = pd.DataFrame(rows); H1.to_csv(f'{OUT}/position_end_vs_panel.csv', index=False)
print('=== Table 15 (end) against Table 9: the two cached score files on the same 210 attacked records (hosts were not rescored in notebook 23; thresholds are Table 9\'s) ==='); print(H1.to_string(index=False))
note('Stage H: largest score difference between the two runs on records whose text is identical: ' + ', '.join(f'{r.detector} {r.max_abs_diff_identical_texts:.2e}' for r in H1.itertuples() if not np.isnan(r.max_abs_diff_identical_texts)) + '; where the texts are identical and the scores still differ, the scorer is not run-to-run stable at that size')

## Stage I. Section 6.2 by ranks

In [ ]:
# notebook 08 built its conditions deterministically: for each payload (harmful, then benign_hijack), each of up to 20 instructions standalone, then embedded in the same 5 hosts; then the 5 clean hosts
CM = pd.read_csv('reports/phase2_cell_means.csv'); P2 = {t: np.load(f'data/phase2/score_{t}.npy') for t in TAGS3 if os.path.exists(f'data/phase2/score_{t}.npy')}
n_h = int(CM[(CM.payload == 'harmful') & (CM.structure == 'standalone')].n.iloc[0]); n_j = int(CM[(CM.payload == 'benign_hijack') & (CM.structure == 'standalone')].n.iloc[0]); n_hosts = int(CM[CM.structure == 'clean_doc'].n.iloc[0])
lay = []
for payload, n_ in (('harmful', n_h), ('benign_hijack', n_j)):
    for k in range(n_):
        lay.append((payload, 'standalone', k, -1)); lay += [(payload, 'embedded', k, h) for h in range(n_hosts)]
lay += [('none', 'clean_doc', -1, h) for h in range(n_hosts)]
LAY = pd.DataFrame(lay, columns=['payload', 'structure', 'instruction', 'host'])
for t, p in P2.items():
    assert len(p) == len(LAY), (t, len(p), len(LAY))
    for (pl, st), g in LAY.groupby(['payload', 'structure']):      # the layout is right only if every cell mean comes back
        ref = CM[(CM.detector == t) & (CM.payload == pl) & (CM.structure == st)]
        check(len(ref) == 1 and abs(float(p[g.index].mean()) - float(ref.mean_p.iloc[0])) < 0.002, t, pl, st, float(p[g.index].mean()), ref.mean_p.values)
note('Stage I: the notebook 08 layout is reconstructed and every cell mean of reports/phase2_cell_means.csv comes back from the cached scores')
rows = []
for t, p in P2.items():
    nm = LABEL[t]; hosts778 = P3[t]['bipia_cat'][HOST]; clean5 = p[LAY.structure == 'clean_doc']
    r = dict(detector=nm, n_hijack_instructions=n_j, n_harmful_instructions=n_h, n_hosts=n_hosts)
    for pl, tag in (('benign_hijack', 'hijack'), ('harmful', 'harmful')):
        emb = p[(LAY.payload == pl) & (LAY.structure == 'embedded')]; alone = p[(LAY.payload == pl) & (LAY.structure == 'standalone')]
        r[f'AUROC_{tag}_embedded_vs_5_clean_hosts'] = round(roc_auc_score(np.r_[np.ones(len(emb)), np.zeros(len(clean5))], np.r_[emb, clean5]), 3)
        r[f'AUROC_{tag}_embedded_vs_778_hosts_other_run'] = round(roc_auc_score(np.r_[np.ones(len(emb)), np.zeros(len(hosts778))], np.r_[emb, hosts778]), 3)
        r[f'AUROC_{tag}_standalone_vs_embedded'] = round(roc_auc_score(np.r_[np.ones(len(alone)), np.zeros(len(emb))], np.r_[alone, emb]), 3)
        # paired by instruction: the embedded copy against the same instruction alone, over every (instruction, host) pair
        E_ = LAY[(LAY.payload == pl) & (LAY.structure == 'embedded')]; SA_idx = {row.instruction: i for i, row in LAY[(LAY.payload == pl) & (LAY.structure == 'standalone')].iterrows()}
        below = [bool(p[i] < p[SA_idx[row.instruction]]) for i, row in E_.iterrows()]
        r[f'share_{tag}_embedded_below_its_standalone'] = fmt(np.mean(below), *wilson(int(sum(below)), len(below)))
        r[f'median_host_percentile_{tag}_embedded'] = round(float(np.median([(hosts778 > v).mean() for v in emb])), 3)
    rows.append(r)
I1 = pd.DataFrame(rows); I1.to_csv(f'{OUT}/phase2_ranks.csv', index=False)
print('=== Section 6.2 by ranks: AUROCs and paired comparisons that a recalibration cannot move (the 778-host comparison crosses two scoring runs; Stage H says how far those runs differ) ==='); print(I1.T.to_string())
note('Stage I: embedded benign-worded injection against the 778 hosts, AUROC: ' + ', '.join(f"{r.detector} {r.AUROC_hijack_embedded_vs_778_hosts_other_run}" for r in I1.itertuples()) + '; share of embedded copies scoring below the same instruction alone: ' + ', '.join(f"{r.detector} {r.share_hijack_embedded_below_its_standalone.split()[0]}" for r in I1.itertuples()))

## Stage J. Figure 8 regenerated, summary, commit

In [ ]:
import matplotlib.pyplot as plt
ev = pd.read_csv('reports/phase5v2_evasion.csv'); mf = pd.read_csv('reports/phase5v2_manufactured.csv')
ev = ev[ev.threshold == 'doc'].reset_index(drop=True); mf = mf[mf.threshold == 'doc'].reset_index(drop=True); assert list(ev.detector) == list(mf.detector)
x = np.arange(len(ev)); w = 0.38; fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].bar(x - w / 2, ev.FNR_static, w, label='static'); ax[0].bar(x + w / 2, ev.FNR_adapt, w, label='adaptive (min of 5 variants)')
ax[0].set_ylabel('FNR at document-calibrated 1 percent FPR'); ax[0].set_title('Evasion'); ax[0].set_ylim(0, 1.05); ax[0].legend(); ax[0].grid(axis='y', alpha=.3)
ax[1].axhline(float(mf.orig_ASR.iloc[0]), color='k', ls=':', label='original attacks: success rate')
ax[1].bar(x - w / 2, mf.still_works, w, label='selected rewrite still works downstream'); ax[1].bar(x + w / 2, mf.manufactured_rate, w, label='manufactured miss (passes and works)')
ax[1].set_ylabel('rate over 91 attacks'); ax[1].set_title('Functionality and manufactured misses'); ax[1].set_ylim(0, 1.05); ax[1].legend(); ax[1].grid(axis='y', alpha=.3)
for a in ax: a.set_xticks(x); a.set_xticklabels(ev.detector, rotation=20, ha='right')
plt.tight_layout(); plt.savefig('figures/phase5v2.png', dpi=150, bbox_inches='tight'); plt.show()
note('Stage J: figures/phase5v2.png regenerated from reports/phase5v2_evasion.csv and reports/phase5v2_manufactured.csv (legend: manufactured miss)')
open(f'{OUT}/summary.txt', 'w').write('\n'.join(SUMMARY) + '\n')
from reslog import log_result
log_result('nb52b: part-one fixes from the cached files (hard review of 10 October), no GPU', '\n'.join(SUMMARY))
print('\n'.join(SUMMARY)); print('\nfiles:', sorted(os.listdir(OUT)))

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb52b: part-one fixes from the cached files, no GPU (hard review of 10 October): Section 8 re-scored under the paper's final rule with string-level bootstrap and host retention; PIGuard and the conventional fine-tune on the 455 cached variants; the 3B probe at all three layers and the cross-tab split into originals and rewrites; a length-only score on the second source; AgentDojo duplicates, tied top scores and the 66 excluded records; Section 5.2 subsampled within deepset; interval ends and rank severity for every detector on both document sets; the two structural ablations audited; Table 15 against Table 9; Section 6.2 by ranks; Figure 8 regenerated; reports/part1_fixes/*"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)